# faster-whisper — Transcription

CTranslate2 Whisper (int8 on CPU). Size is your speed/accuracy dial: tiny -> large-v3.

**Model:** `faster-whisper`  |  **Feature:** Speech-to-text / subtitles

Runs on CPU or GPU. Cells: setup → data → load → test → benchmark → accuracy → training notes → copyright.


## 1. Setup - code + dependencies


In [ ]:
import os, sys, subprocess, time, io, urllib.request, platform
from pathlib import Path
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists(): ROOT = Path('.').resolve()
elif Path('../app').exists(): ROOT = Path('..').resolve()
else:
    subprocess.run(['git','clone','--depth','1',REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','pillow','numpy','onnxruntime','soundfile'], check=False)
print('root:', ROOT, '| python', platform.python_version())

## 2. Data (test set)


In [ ]:
import numpy as np, soundfile as sf
# Synthetic tone has no words; use a REAL speech clip for meaningful WER.
AUDIO = ''  # path to a speech wav/mp3; leave empty to just smoke-test the load
print('audio set:', bool(AUDIO))

## 3. Download & load the model


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','faster-whisper'], check=False)
SIZE = 'base'  # tiny/base/small/medium/large-v3
os.environ['SHORTS_WHISPER_MODEL']=SIZE
print('faster-whisper', SIZE, '- weights download on first transcription')

## 4. Inference test


In [ ]:
from app.shorts_pipeline import transcribe_audio
from pathlib import Path as _P
if AUDIO and _P(AUDIO).exists():
    tr = transcribe_audio(_P(AUDIO), provider='local', model_size=SIZE)
    print(tr.engine, '|', tr.language, '|', len(tr.segments), 'segments')
    print(tr.full_text[:400])
else: print('Set AUDIO to a real speech file to transcribe.')

## 5. Benchmark (latency)


In [ ]:
from app.shorts_pipeline import transcribe_audio
from pathlib import Path as _P
if AUDIO and _P(AUDIO).exists():
    import time; t=time.perf_counter(); tr=transcribe_audio(_P(AUDIO), provider='local', model_size=SIZE)
    import soundfile as sf; dur=len(sf.read(str(_P(AUDIO)))[0])/sf.info(str(_P(AUDIO))).samplerate
    print(f'{time.perf_counter()-t:.1f}s to transcribe {dur:.0f}s -> {dur/(time.perf_counter()-t):.1f}x realtime')
else: print('need an audio file')

## 6. Accuracy signal


In [ ]:
print('Reported WER (LibriSpeech test-clean): tiny ~7.5%, base ~5-6% (English).')
print('Numbers vary by dataset/language; measure with jiwer against a reference transcript:')
print('  pip install jiwer; from jiwer import wer; wer(reference, hypothesis)')

## 7. Training / fine-tuning notes

Whisper was trained by OpenAI on ~680k hours of multilingual audio; faster-whisper is a CTranslate2 conversion for fast inference. You would not retrain it - you pick a size (tiny..large-v3) to trade speed against accuracy, or fine-tune upstream Whisper on a GPU.


## 8. Copyright


In [ ]:
# ====================================================================
# COPYRIGHT NOTICE - DO NOT REMOVE
# ====================================================================
# (c) Zubair Hussain - Xovato AI Studio.
# These notebooks, the benchmark/accuracy harness, and the surrounding
# tooling are the work of Zubair Hussain. You may use and modify them for
# your own projects; keep this attribution intact, including in the
# background-removal notebook and any derived outputs.
print('(c) Zubair Hussain - Xovato AI Studio')